In [ ]:
# ===== E3 — Qwen3-VL-Embedding hi-res (GPU, ~৩-৪ ঘণ্টা) =====  ★ মূল বাজি ★
# কাজ: যে encoder প্রকল্পের সবচেয়ে বড় লাফ দিয়েছিল (0.4806 -> 0.62319), তাকে
#      thumbnail-এর বদলে আসল ছবি খাওয়ানো।
#
# আবিষ্কার: Qwen3-VL naive-dynamic-resolution model — যত pixel, তত visual token।
#   এখন সে পায় 384x225  -> ~১১২ token
#   আসল 2066x1209 হলে   -> ~৩,১৮২ token   (≈২৮ গুণ)
#   পুরো repo-তে min_pixels/max_pixels কোথাও সেট নেই — অর্থাৎ cache-ই একমাত্র
#   বাধা ছিল, model নয়। সে তার ইনপুট ক্ষমতার ~৪%-এ চলছিল।
#
# আমরা 1280px cache + max_pixels ≈ 1.0 Mpx (~১২৮০ token) ব্যবহার করব — VRAM-এ
# যতটা কুলোয়। ১১২ -> ~১২৮০, প্রায় ১১ গুণ।
#
# ⚠️ ফাইলের নাম: emb_qvl8_img.npy (emb_qvl_img.npy-এর উপরে লিখি না)।
#    plan_d_final-এ qvl8 একটা আলাদা খালি space — পুরনো আর নতুন পাশাপাশি বসবে,
#    LightGBM নিজেই বেছে নেবে। এতে A/B পরিষ্কার থাকে, আর find() ভুল ফাইল
#    ধরার ঝুঁকিও থাকে না। plan_d_final-এ কোনো কোড বদলাতে হবে না।
#
# Accelerator: GPU T4 x2 / P100।  Internet: ON (model নামাতে)।
# Input: img1280 (E0-র output) + essentials + result_plan_d_a (emb_qvl_txt.npy-র জন্য)
import os, glob, time, gc, sys, subprocess, shutil
import numpy as np, pandas as pd
OUT, INP = '/kaggle/working', '/kaggle/input'
T0 = time.time()
def tlog(*a): print(f'[{(time.time()-T0)/60:6.1f} min]', *a, flush=True)
def find(n, isdir=False):
    for r in (INP, OUT):
        for h in glob.glob(f'{r}/**/{n}', recursive=True):
            if os.path.isdir(h) == isdir: return h
    return None
def nrm(x):
    x = np.asarray(x, dtype=np.float32)
    return x / np.linalg.norm(x, axis=1, keepdims=True).clip(1e-8)

subprocess.run(f'{sys.executable} -m pip install -q -U "sentence-transformers>=5" qwen-vl-utils',
               shell=True, timeout=1800)
import torch
tlog('VRAM', f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB')

IMGD = find('img1280', isdir=True)
OLDD = find('img384',  isdir=True)
assert IMGD, 'img1280 নেই — আগে E0 চালিয়ে dataset বানাও'
IH = pd.read_parquet(find('img_hashes.parquet')).hash.astype(str).values
have = {f[:-4] for f in os.listdir(IMGD) if f.endswith('.jpg')}
assert not (set(IH) - have), f'{len(set(IH)-have)}টা ছবি img1280-এ নেই — E0 শেষ করো'
tlog(f'ছবি {len(IH)} | img1280 {IMGD}')

In [ ]:
# ===== E3 — CELL 1 : model + max_pixels + 🚨 token যাচাই =====
from sentence_transformers import SentenceTransformer
MODEL = 'Qwen/Qwen3-VL-Embedding-2B'
MAX_PIXELS = 1280 * 28 * 28        # ≈1.00 Mpx → ~১২৮০ visual token
MIN_PIXELS = 256 * 28 * 28

model = SentenceTransformer(MODEL, device='cuda', model_kwargs={'torch_dtype': torch.float16})
tlog('model loaded')

# ---- image processor খুঁজে বের করা (ST-এর ভেতরের গঠন version-ভেদে বদলায়) ----
def find_ip(m):
    seen = []
    for mod in list(m) + [m]:
        for attr in ('processor', 'image_processor'):
            o = getattr(mod, attr, None)
            if o is None: continue
            ip = getattr(o, 'image_processor', o)
            if hasattr(ip, 'max_pixels') or hasattr(ip, 'size'):
                seen.append((f'{type(mod).__name__}.{attr}', ip))
    return seen

cands = find_ip(model)
print('পাওয়া processor:', [c[0] for c in cands] or 'কিছু না')
assert cands, 'image processor পাওয়া গেল না — নিচের probe ছাড়া এগিয়ো না'

for name, ip in cands:
    before = (getattr(ip,'min_pixels',None), getattr(ip,'max_pixels',None), getattr(ip,'size',None))
    for k, v in (('min_pixels',MIN_PIXELS), ('max_pixels',MAX_PIXELS)):
        try: setattr(ip, k, v)
        except Exception as e: print('  সেট করা গেল না', k, repr(e)[:80])
    if isinstance(getattr(ip,'size',None), dict):
        ip.size = {**ip.size, 'shortest_edge': MIN_PIXELS, 'longest_edge': MAX_PIXELS}
    print(f'  {name}: {before} -> ({ip.min_pixels}, {ip.max_pixels}, {getattr(ip,"size",None)})')

# ---- 🚨 আসল যাচাই: token সত্যিই বেড়েছে কিনা। ৪ ঘণ্টা পরে নয়, এখনই। ----
from PIL import Image
_, ip0 = cands[0]
ms = getattr(ip0, 'merge_size', 2)
def ntok(path):
    im = Image.open(path).convert('RGB')
    out = ip0(images=im, return_tensors='pt')
    g = out.get('image_grid_thw')
    if g is None: return im.size, None
    g = g[0].tolist()
    return im.size, int(np.prod(g) // (ms*ms))

h0 = IH[0]
sz_new, tk_new = ntok(f'{IMGD}/{h0}.jpg')
print(f'\n{"":8s} {"pixel আকার":>16s} {"visual token":>14s}')
if OLDD and os.path.exists(f'{OLDD}/{h0}.jpg'):
    sz_old, tk_old = ntok(f'{OLDD}/{h0}.jpg')
    print(f'{"পুরনো":8s} {str(sz_old):>16s} {str(tk_old):>14s}')
print(f'{"নতুন":8s} {str(sz_new):>16s} {str(tk_new):>14s}')

assert tk_new and tk_new > 400, (
    f'token মাত্র {tk_new} — max_pixels কার্যকর হয়নি বা ছবি ছোট। '
    'এই অবস্থায় ৪ ঘণ্টা চালানো পুরো অপচয়। থামো।')
print(f'\n✅ token {tk_new} — hi-res কার্যকর হয়েছে, এগোনো যাবে')

t = time.time(); _ = model.encode([{'image': f'{IMGD}/{h}.jpg'} for h in IH[:4]], batch_size=2)
per = (time.time()-t)/4
print(f'গতি {per:.2f}s/ছবি → {len(IH)}টায় ≈ {per*len(IH)/60:.0f} মিনিট')

In [ ]:
# ===== E3 — CELL 2 : image embedding (checkpoint-সহ) =====
BS, BUDGET, CK = 2, 10.0*3600, f'{OUT}/_ck_qvl8_img.npy'
items = [{'image': f'{IMGD}/{h}.jpg'} for h in IH]

done = []
p = CK if os.path.exists(CK) else find('_ck_qvl8_img.npy')
if p:
    done = list(np.load(p)); tlog(f'checkpoint থেকে {len(done)}')

t0 = time.time()
for i in range(len(done), len(items), 500):
    done.extend(model.encode(items[i:i+500], batch_size=BS, show_progress_bar=False))
    np.save(CK, np.array(done, dtype=np.float32))
    el = time.time()-t0; n = max(len(done)-i, 1)
    tlog(f'  {len(done)}/{len(items)} | বাকি ~{(len(items)-len(done))*el/max(len(done)-i,1)/60:.0f}m')
    if time.time()-T0 > BUDGET:
        tlog('⏱️ সময়সীমা — Output→Dataset করে আবার চালাও'); break

A = np.array(done, dtype=np.float32)
if len(A) < len(items):
    print(f'⚠️ {len(items)-len(A)}টা বাকি — এখন গড় vector দিয়ে ভরছি (feature নিরপেক্ষ থাকবে)।')
    print('   সম্পূর্ণ করতে notebook আবার চালাও; অসম্পূর্ণ অবস্থায় plan_d_final-এ দিয়ো না।')
    A = np.vstack([A, np.tile(A.mean(0), (len(items)-len(A), 1))])
A = nrm(A)
np.save(f'{OUT}/emb_qvl8_img.npy', A)
tlog(f'emb_qvl8_img.npy সেভ {A.shape}')
del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ===== E3 — CELL 3 : text দিক + যাচাই =====
# text বদলায়নি (caption আগেও পরিষ্কার ছিল), তাই পুরনোটাই qvl8_t হিসেবে কপি করি।
# plan_d_final image+text-এ qvl8 ব্যবহার করে শুধু যদি দুই অর্ধই থাকে।
pt = find('emb_qvl_txt.npy')
if pt:
    shutil.copy(pt, f'{OUT}/emb_qvl8_txt.npy')
    T = np.load(f'{OUT}/emb_qvl8_txt.npy')
    print(f'✅ emb_qvl8_txt.npy = পুরনো emb_qvl_txt.npy-র কপি {T.shape}')
else:
    print('🚨 emb_qvl_txt.npy পাওয়া গেল না — result_plan_d_a attach করো।')
    print('   qvl8_t ছাড়া image+text-এ qvl8 ব্যবহারই হবে না (image+image-এ হবে)।')

A = np.load(f'{OUT}/emb_qvl8_img.npy')
print(f'\nemb_qvl8_img {A.shape} | L2 norm গড় {np.linalg.norm(A,axis=1).mean():.4f}')

old = find('emb_qvl_img.npy')
if old:
    B = nrm(np.load(old))
    if B.shape == A.shape:
        cs = (A*B).sum(1)
        print(f'পুরনো vs নতুন cosine: গড় {cs.mean():.3f} | মধ্যক {np.median(cs):.3f} '
              f'| সর্বনিম্ন {cs.min():.3f}')
        print('  (১.০-র খুব কাছে হলে hi-res আসলে কিছু বদলায়নি — সন্দেহজনক।')
        print('   ০.৭-০.৯৫ মানে সত্যিই নতুন তথ্য ঢুকেছে।)')

idx = np.arange(0, len(A), max(1, len(A)//2000))
S = A[idx] @ A[idx].T; np.fill_diagonal(S, -1)
print(f'\nনমুনা cosine — গড় {S.mean():.3f} | top-1 গড় {S.max(1).mean():.3f}')
print('  (top-1 গড় যত বেশি গড়ের চেয়ে, space তত ভালো বৈষম্য করছে)')

print('\n👉 Save Version → Output কে Dataset বানাও (নাম: qvl8-hires)।')
print('   plan_d_final-এ attach করলে CELL 0-এ "space qvl8_i" ও "space qvl8_t" দেখা যাবে।')
tlog('done')